# Cyclical epidemic dynamics

Adapted from the [Monash EMU summer textbook](https://github.com/monash-emu/summer-textbook)
notebook `textbook/11-cyclical-epidemics.ipynb` at commit
`fd97783474789e50ace5ea420aec20147f9bbd76`.

Source licence: BSD-2-Clause, Copyright (c) 2022, monash-emu. Prose is carried
and adapted; code is written in summer4 idiom.

Epidemics slow as susceptibles are depleted. If susceptibles are later
replenished — by waning immunity or by births — transmission can rise again and
produce **cyclical** waves even when every parameter is constant. This chapter
shows damped oscillation under slow waning, the same pattern under
replacement births, and a phase-plane view of the approach to endemicity.


## Waning immunity

Start from an SIR model with frequency-dependent infection, then add a slow
recovered → susceptible flow. The first wave is large; later waves are smaller
as the system spirals toward an endemic equilibrium (damped oscillation).


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

from summer4 import (
    Compartments,
    EntryFlow,
    Everything,
    ExitFlow,
    Property,
    PropertyMap,
    SavePlan,
    SaveRequest,
    FlowModel,
    TransitionFlow,
)
from summer4.epi import ForceOfInfection, MixingMatrix

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

state = Property("state", ("susceptible", "infectious", "recovered"))
pop = Property("pop", ("all",))
pmap = PropertyMap.from_property(state).stratify(pop)

POPULATION = 1e5
SEED = 1.0
END_TIME = 1e3
PLAN = SavePlan(
    requests={"comp": SaveRequest(Compartments())},
    ts=np.linspace(0.0, END_TIME, int(END_TIME) + 1),
)


def make_y0() -> np.ndarray:
    y0 = np.zeros(pmap.size)
    y0[pmap.select(state["susceptible"])] = POPULATION - SEED
    y0[pmap.select(state["infectious"])] = SEED
    return y0


def compartment_frame(res) -> pd.DataFrame:
    data = {
        name: np.asarray(res["comp"].select(state[name]).values.data)[:, 0]
        for name in ("susceptible", "infectious", "recovered")
    }
    return pd.DataFrame(data, index=np.asarray(res["comp"].times.values))


def local_maxima(series: np.ndarray, *, min_height: float) -> list[int]:
    """Indices of strict local peaks above ``min_height`` (no SciPy)."""
    peaks: list[int] = []
    for i in range(1, len(series) - 1):
        if (
            series[i] > series[i - 1]
            and series[i] >= series[i + 1]
            and series[i] > min_height
        ):
            peaks.append(i)
    return peaks


def base_sir() -> FlowModel:
    model = FlowModel(pmap)
    
    mixing = MixingMatrix(pop, np.array([[1.0]]), check_reciprocal=False)
    model.add_flow(
    TransitionFlow(
        "infection",
        state["susceptible"],
        state["infectious"],
        ForceOfInfection(
    "infection",
    infectious=state["infectious"],
    group_by=pop,
    kind="frequency",
    contact_rate=1.0,
    mixing=mixing,
),
    )
)
    model.add_flow(TransitionFlow(
        "recovery", state["infectious"], state["recovered"], 0.333
    ))
    return model


In [ ]:
wane = base_sir()
wane.add_flow(TransitionFlow(
    "waning", state["recovered"], state["susceptible"], 0.005
))
wane_res = wane.compile().run(
    {}, make_y0(), t0=0.0, t1=END_TIME, dt=0.5, save=PLAN, solver="euler"
)
wane_out = compartment_frame(wane_res)

i_path = np.asarray(wane_out["infectious"].values)
peaks = local_maxima(i_path, min_height=100.0)
assert len(peaks) >= 3, "expect several epidemic cycles under slow waning"
assert i_path[peaks[0]] > i_path[peaks[1]] > i_path[peaks[2]], (
    "damped oscillation: successive peaks should shrink"
)

wane_out.plot(
    title="SIRS with slow waning immunity",
    labels={"index": "time", "value": "compartment size"},
)


A higher contact rate lowers the susceptible fraction needed to start a new
wave, so cycles return sooner. Faster replenishment does the same. The
pattern of a large first wave followed by smaller ones is classic damped
oscillation.

## Demographic replenishment

Another route back to susceptibility is births. Apply a universal death rate
and replace every death with an entry into the susceptible compartment so the
total population stays closed. Using the same numerical rate as the waning
example produces a very similar cycle; the infectious sojourn is slightly
shorter because deaths also remove people from `infectious`, which trims $R_0$
a little.


In [ ]:
demog = base_sir()
death = demog.add_flow(ExitFlow("universal_death", Everything(), 0.005))
demog.add_flow(EntryFlow("births", state["susceptible"], death.sum()))

demog_res = demog.compile().run(
    {}, make_y0(), t0=0.0, t1=END_TIME, dt=0.5, save=PLAN, solver="euler"
)
demog_out = compartment_frame(demog_res)

pop_start = float(demog_out.iloc[0].sum())
pop_end = float(demog_out.iloc[-1].sum())
assert abs(pop_start - POPULATION) < 1e-6
assert abs(pop_end - POPULATION) < 1.0  # Euler mass drift over long horizon

i_demog = np.asarray(demog_out["infectious"].values)
demog_peaks = local_maxima(i_demog, min_height=100.0)
assert len(demog_peaks) >= 3

demog_out.plot(
    title="SIR with replacement births (closed population)",
    labels={"index": "time", "value": "compartment size"},
)


## Phase plane

Plotting susceptibles against infectious people (after the first takeoff)
shows the endemic equilibrium as an attractor the trajectory spirals into.
Each loop overshoots before settling — the time dimension is roughly the
distance along the spiral.


In [ ]:
late = wane_out.loc[wane_out.index > 70.0]
fig = px.line(
    late,
    x="susceptible",
    y="infectious",
    title="Phase plane (waning model, t > 70)",
    labels={"susceptible": "susceptible", "infectious": "infectious"},
)
fig.show()


The path first creeps rightward along the susceptible axis, then the primary
wave arcs through high infectiousness. Subsequent loops tighten around the
interior equilibrium.

A three-dimensional view adds time as the vertical axis:


In [ ]:
fig3d = px.line_3d(
    late,
    x="susceptible",
    y="infectious",
    z=late.index,
    title="Phase plane with time",
    width=800,
    height=600,
    labels={
        "susceptible": "susceptible",
        "infectious": "infectious",
        "z": "time",
    },
)
fig3d.show()


## Summary

- Slow susceptible replenishment (waning or births) can sustain **cyclical**
  epidemics under constant parameters.
- Early waves are larger; later waves damp toward an endemic state.
- The $(S, I)$ phase plane makes that spiral geometry visible.
